<b>Group Number:</b> 8
<br><b>Name Group Member 1: Anton Seifert</b> 
<br><b>u-Kürzel Group Member 1: unjud</b> 
<br><b>Name Group Member 2: Arne Segatz</b> 
<br><b>u-Kürzel Group Member 2: urnye</b> 


# Convolutional Neural Networks (CNNs) - Part 2
In the following, CNNs will be examined by some common tasks in the field of image processing. Be aware that CNNs can be used in various tasks where many datapoints relate locally to each other.

## 2.1 Some Imports and Preparation
The following initalizations might take some time ...

In [1]:
from ipywidgets import widgets
from IPython.display import display

import matplotlib.pyplot as plt
import numpy as np

# For some convolving operations
from scipy import signal

# DeepLearning Library Keras
# Documentation https://keras.io/
import tensorflow as tf
import tensorflow.keras as keras
from tensorflow.keras.datasets import cifar10
from tensorflow.keras.callbacks import TensorBoard, ModelCheckpoint
from tensorflow.keras.layers import Input, Dense, Conv2D, MaxPooling2D, Flatten, BatchNormalization, Dropout, Reshape
from tensorflow.keras.models import Model, Sequential
from tensorflow.keras.preprocessing.image import ImageDataGenerator

import os, sys
from typing import *

from lama.test_functions import CNN_Tests

test_func = CNN_Tests()

# allow memory to grow, not consume all at once
gpus = tf.config.list_physical_devices('GPU')
for gpu in gpus:
    tf.config.experimental.set_memory_growth(gpu, True)

# Size for some plots with matplotlib
figure_inches = 3

## 2.3 Cifar-10 classification task
Beside the MNIST dataset Cifar10 is as well a small dataset used in the beginning of CNNs. There are 10 different classes of simple objects or animals. The images are of size 32x32x3. In this section, you should tune a given CNN in order to classify images with high accuracy. 

See also: [Cifar-10](https://www.cs.toronto.edu/~kriz/cifar.html)


<div class="alert alert-block alert-success">
<b>Task:</b> Load the dataset, define number of classes, transform labels and define all corresponding classes (like airplane,...) according to the comments in the code cells.

</div>

In [2]:
# Load the dataset from Keras, Tip: cifar10 is already imported, train + test set required
x_train: np.ndarray
y_train: np.ndarray
x_test: np.ndarray
y_test: np.ndarray

# STUDENT CODE HERE (1 pts)

(x_train, y_train), (x_test, y_test) = ...

# STUDENT CODE until HERE

In [3]:
# How many classes are in Cifar-10? 
num_classes: int

### STUDENT CODE HERE (1 pts)

num_classes = ...

### STUDENT CODE until HERE

print(num_classes)

In [4]:
# Transform the labels into categorical vectors
# Use the keras.utils.to_categorical function
y_train_categorical: np.ndarray
y_test_categorical: np.ndarray

### STUDENT CODE HERE (1 pts)

y_train_categorical = ...
y_test_categorical = ...

### STUDENT CODE until HERE

In [5]:
# What classes are there? Define them in a list of strings named classes.
classes: List[str]

### STUDENT CODE HERE (2 pts)

classes = ...

### STUDENT CODE until HERE

# Sanity check - compare your results
test_func.test_classes(classes)

<div class="alert alert-block alert-success">
<b>Task:</b> Check if you defined everything as required.

</div>

In [6]:
# Image in the training set
number_sample = 6  # test multiple ones

fig, ax = plt.subplots(figsize=(figure_inches, figure_inches))
ax.set_title(classes[y_test[number_sample].item()]+' ?', fontsize = 15)
ax.imshow(x_test[number_sample,:,:,:], interpolation='nearest')
plt.tight_layout()


<div class="alert alert-block alert-success">
<b>Task:</b> Find a picture of a horse (not by try and error) and plot it using the code above. Use the code cell below.

</div>

In [7]:
### STUDENT CODE HERE (1 pts)


### STUDENT CODE until HERE


<div class="alert alert-block alert-success">
<b>Task:</b> Preprocess the data to ensure values between 0 and 1 by dividing rgb values by their maximum value.

</div>

In [8]:
# Data Preprocessing

### STUDENT CODE HERE (1 pts)

x_train = ...
x_test = ...

### STUDENT CODE until HERE

<div class="alert alert-block alert-success">
<b>Question (1 pts):</b> How many training and test samples are there? 
</div>

<div class="alert alert-block alert-success">
<b>Your Answer:</b> 
</div>

<div class="alert alert-block alert-success">
<b>Question (2 pts):</b> Why normalize the data?
</div>

<div class="alert alert-block alert-success">
<b>Your Answer:</b> 
</div>

<div class="alert alert-block alert-success">
<b>Question (2 pts):</b> Why using a categorical vector instead of a single output?
</div>

<div class="alert alert-block alert-success">
<b>Your Answer:</b> 
</div>

## 2.4 Classification models for CIFAR-10

### 2.4.1 Neural Network Classifier:

In [9]:
def model_nn() -> tf.keras.Model:
    return Sequential([
        Input(shape = x_train.shape[1:], name='Input_MLP'),
        Flatten(name='Flattening_MLP'),
        Dense(256, activation = 'relu', name='Hidden1_NN'),
        Dense(256, activation = 'relu', name='Hidden2_NN'),
        Dense(num_classes, activation = 'softmax', name='Output_NN')
    ])

### 2.4.2 Convolutional Neural Network Classifier:

In [10]:
def model_cnn() -> tf.keras.Model:
    return Sequential([
        Input(shape = x_train.shape[1:], name='Input_MLP'),
        Conv2D(filters= 16, kernel_size = (3,3), padding='same', activation = 'relu', name='Conv1'),
        MaxPooling2D(pool_size = (2,2), strides = (2,2), padding='valid', name='Pool1'),
        Conv2D(filters = 32, kernel_size = (3,3), padding='same', activation = 'relu', name='Conv2'),
        MaxPooling2D(pool_size = (2,2), strides = (2,2), padding='valid', name='Pool2'),
        Flatten(name='Flatt_CNN'),
        Dense(256, activation = 'relu', name='FC-1'),
        Dense(num_classes, activation = 'softmax', name='Output_CNN')
    ])

### 2.4.3 Comparison of MLP and CNN Classifiers:


<div class="alert alert-block alert-success">
<b>Task:</b> In order to classify the images in Cifar-10, use the given MLP and CNN models to examine which one performs better.
Train both networks for 10 epochs and look at the results.
Feel free to use and change the code in the two code cells down below. If your network does not train, you might have not prepared the rgb-values in the right way (For example: You did not normalize or you did it too often).
</div>

<div class="alert alert-block alert-info">
<b>Note:</b> Structure of code cells below
<ul>
<li> Use the predefined functions to create your model
<li> Define the common TensorBoard logger with the configuration to look at training results later on
<li> Compile and fit the model
<li> Hint: If your models do not learn anything, check your data normalization. You might have not normalized your data or too often.
</li>


</ul>


</div>

In [11]:
# Method for plotting the accuracy history from the model training history
def plot_metric_history(history: tf.keras.callbacks.History, metric: str):
    n_epochs = len(history[metric])

    plt.plot(range(1, n_epochs + 1), history[metric], label=f'train {metric}')
    plt.plot(range(1, n_epochs + 1), history[f'val_{metric}'], label=f'validation {metric}')
    plt.legend()
    plt.title(f'Training and validation {metric} for {n_epochs} epochs of training.')
    plt.show()

In [12]:
# Train the Neural Network (MLP)

nn_model = model_nn()
config_nn = 'UNRECOGNIZEABLE_NAME_EDIT_ME_PLEASE' # Give a recognizable name

# The TensorBoard is a feature of tensorflow for the visualization of the training process 
nn_logger = TensorBoard(log_dir='logs/nn_logs/'+config_nn+'/')

nn_model.compile(loss='categorical_crossentropy', metrics=['accuracy'], optimizer='Adam')
history = nn_model.fit(x_train, y_train_categorical, batch_size=64, epochs=10, 
                       validation_data=(x_test, y_test_categorical), 
                       callbacks=[nn_logger], verbose=1)
plot_metric_history(history.history, 'accuracy')

<div class="alert alert-block alert-success">
<b>Task:</b> Use TensorBoard to control your training progress. An explanation on how to open your TensorBoard is given here:
    <a href="https://github.com/tensorflow/tensorboard/blob/master/docs/r1/summaries.md">TensorBoard</a>  (at the bottom of the webpage)
</div>

In [13]:
# Train the CNN

cnn_model = model_cnn()
config_cnn = 'UNRECOGNIZEABLE_NAME_EDIT_ME_PLEASE' # give a recognizable name

cnn_logger = TensorBoard(log_dir='logs/cnn_logs/'+config_cnn+'/') 

cnn_model.compile(loss='categorical_crossentropy', metrics = ['accuracy'], optimizer='Adam')
history = cnn_model.fit(x_train, y_train_categorical, batch_size=64, epochs=10, 
                        validation_data=(x_test, y_test_categorical), 
                        callbacks=[cnn_logger], verbose=1)
plot_metric_history(history.history, 'accuracy')

<div class="alert alert-block alert-success">
<b>Question (1 pts):</b> Which network performs better?
</div>

<div class="alert alert-block alert-success">
<b>Your Answer:</b> 
</div>

<div class="alert alert-block alert-success">
<b>Question (1 pts):</b> How many parameters do the networks have? Therefore use the summary method (see Keras-Docs)...
</div>

<div class="alert alert-block alert-success">
<b>Your Answer:</b> 
</div>

<div class="alert alert-block alert-success">
<b>Question (2 pts):</b> Where are most parameters stored in this CNN?
</div>

<div class="alert alert-block alert-success">
<b>Your Answer:</b> 
</div>

### 2.4.4 Challenge: Optimize the Network! 

<div class="alert alert-block alert-success">
<b>Task:</b> Try to improve one of the models so that your validation accuracy is higher than 0.75 percent once!

<ul>
<li>Hint: Try to overfit first and then regularize.
<li>Hint2: Therefore use L1/L2 - regularization and/or Dropout. BatchNormalization might improve things as well. Look therefore at Keras website for examples or ask tutors.
<li>Hint3: Use one of the functions <code>def model_nn()</code> or <code>def model_cnn()</code> from above.
<li> Hint4: You are also allowed to increase the number of epochs. Have fun and good Luck!

</li>
</ul>
</div>

In [14]:
# Use this block to train the optimized Network.
# You can copy and paste code from above
# You might also want to look online for good training strategies

### STUDENT CODE HERE (8 pts)


### STUDENT CODE until HERE


<div class="alert alert-block alert-success">
<b>Question (2 pts):</b> Describe briefly what you did to improve your network? Name two things.
</div>

<div class="alert alert-block alert-success">
<b>Your Answer:</b> 
</div>

## 2.5 Data Augmentation

Another way to regularize your network is to augment the training data. Use therefore the ImageDataGenerator from Keras. We will later shift and rotate images by ourselves after optimizing on Cifar-10.

In [15]:
# Keras ImageDataGenerator
datagen = ImageDataGenerator(
    featurewise_center=False,
    featurewise_std_normalization=False,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    horizontal_flip=True)

### 2.5.1 Even more challenging ((Bonus Question :)))

<div class="alert alert-block alert-success">
<b>Task:</b> Improve your model and adapt it, how accurate can it get now?
Our solution is able to reach 0.8894 on the validation accuracy.
</div>

In [16]:
def model_cnn_aug() -> tf.keras.Model:
    return Sequential([
        Input(shape=x_train.shape[1:], name='Input_MLP'),
        Conv2D(filters=16, kernel_size=(3,3), padding='same', activation='relu', name='Conv1'),
        MaxPooling2D(pool_size=(2,2), strides=(2,2), padding='valid', name='Pool1'),
        Conv2D(filters=32, kernel_size=(3,3), padding='same', activation='relu', name='Conv2'),
        MaxPooling2D(pool_size=(2,2), strides=(2,2), padding='valid', name='Pool2'),
        Flatten(name='Flatt_CNN'),
        Dense(256, activation='relu', name='FC-1'),
        Dense(num_classes, activation='softmax', name='Output_CNN')
    ])

In [17]:
# Train your model that makes use of data augmentation
# Fit the training data to the data-generator
datagen.fit(x_train)

# Train your CNN_augmentation model:
cnn_aug_model = model_cnn_aug()
cnn_aug_model.compile(loss='categorical_crossentropy', metrics=['accuracy'], optimizer='Adam')
config_cnn_aug = 'None' # give a recognizable name
cnn_logger = TensorBoard(log_dir='logs/cnn_aug_logs/'+config_cnn_aug+'/') 

history = cnn_aug_model.fit(datagen.flow(x_train, y_train_categorical, batch_size=128),
                            epochs=10, validation_data=(x_test, y_test_categorical),
                            callbacks=[cnn_logger], verbose=1)
plot_metric_history(history.history, 'accuracy')

<div class="alert alert-block alert-success">
<b>Question (1 pts):</b> Can you imagine why the ground-truth labels were not augmented in the code cell above and might that be necessary? If the intuition is missing you might come back to this question after you finished the notebook or the implementation of data augmentation down below.  
</div>

<div class="alert alert-block alert-success">
<b>Your Answer:</b> 
</div>

<div class="alert alert-block alert-success">
<b>Question (1 pts):</b> What do you think happens to images adapted by the DataGenerator?
</div>

<div class="alert alert-block alert-success">
<b>Your Answer:</b> 
</div>

## 2.6 Predict with your model

In [18]:
# Get the probabilities of one image prediction
x_tester = x_test[0,:,:,:]

# Use numpy expand_dims before predicting with your model
# Print your predicted classes for the first test image (x_test[0,:,:,:])

# STUDENT CODE HERE (2 pts)

pred = ...

# STUDENT CODE until HERE


<div class="alert alert-block alert-success">
<b>Question (1 pts):</b> With how much confidence was image <b>18</b> (index 17) in the test set predicted as a bird by your model?
</div>

<div class="alert alert-block alert-success">
<b>Your Answer:</b> 
</div>

## 2.7 Let's have a closer look / Get the weights in a convolutional layer

In [19]:
# Get the weights of a layer of one of your models, you specified by name
layer_visual = cnn_model.get_layer('Conv1')
weights = layer_visual.get_weights()[0]

# Take some of them, last dimension are the channels
weights_2d = weights[:,:,0,0] # filters are [:,:, dimension of spatial input (e.g.: rgb=3), nb_filters] in a layer
weights_2d

<div class="alert alert-block alert-success">
<b>Task:</b> Look at layer Conv2 (or another layer than conv1) and plot one filter-kernel-slice of it's 3rd filter. Hint: Use weights.shape to understand the kernel's dimensions.
</div>

In [20]:
# Copy and paste necessary code for this Task from above.
### STUDENT CODE HERE (1 pts)


### STUDENT CODE until HERE


<div class="alert alert-block alert-success">
<b>Question (2 pts):</b> Explain what the dimensions a,b,c and d are in 'weights[a,b,c,d]' like it is used in the code block above. 
</div>

<div class="alert alert-block alert-success">
<b>Your Answer:</b> 

</div>

In [21]:
#Load Ascent image from scipy and convolve it with the previous loaded filter
import scipy.datasets
import scipy.signal

ascent = scipy.datasets.ascent()
ascent = scipy.signal.convolve2d(ascent, weights_2d, boundary='symm', mode='same')
# ascent = np.maximum(ascent, 0)
fig, ax = plt.subplots(figsize=(figure_inches, figure_inches))
ax.imshow(ascent, interpolation='nearest', cmap='gray')
plt.tight_layout()

<div class="alert alert-block alert-success">
<b>Task:</b> Use different filters on the input image. Can you notice any differences? (A few words)
</div>

In [22]:
# Copy and paste the code from above. And use different filters.

# STUDENT CODE HERE (3 pts)


# STUDENT CODE until HERE

<div class="alert alert-block alert-success">
<b>Question (2 pts):</b> Explain briefly what you changed in the pasted code.
</div>

<div class="alert alert-block alert-success">
<b>Your Answer:</b> 
</div>

## 2.8 Visualize the activation in a feedforward pass

In the following code we will directly use the output of the convolutional layer in the CNN and visualize it. This is approximately the same as we did above.  

In [23]:
# Get the output in a feedforward process from a model with get_output function
number_sample = 9

def get_output(model, layer_name, model_input):
    """Function to get activations of a given layer after a feedforward pass.
    
    This approach works with Sequential models that may not have been built yet.
    """
    # Create a new functional model by reconstructing layers up to target layer
    input_layer = keras.Input(shape=model_input.shape[1:])
    x = input_layer
    
    # Apply each layer up to and including the target layer
    for layer in model.layers:
        x = layer(x)
        if layer.name == layer_name:
            break
    
    # Create the partial model
    partial_model = keras.Model(inputs=input_layer, outputs=x)
    return partial_model(model_input)

# Model and layer where the feature maps come from
feature_map = get_output(cnn_model, 'Conv1', np.expand_dims(x_test[number_sample,:,:,:],axis=0))

# First print sample
plt.imshow(x_test[number_sample,:,:,:])
plt.title(classes[y_test[number_sample].item()])

# Now plot all filters
fig, ax = plt.subplots(nrows=4, ncols=4, figsize=(10,10))
axes = ax.flatten()

# Note: it only works for 16 filters, if you want to doodle nrows * ncols
for i, ax in enumerate(axes):
    ax.imshow(feature_map[0,:,:,i], cmap='gray')
    ax.set_title(f"Filter: {i}")

fig.tight_layout()
plt.show()

## Further Reading

[SegmentationForAutonomousDriving](https://blog.playment.io/semantic-segmentation-models-autonomous-vehicles/#U-Net)

[Dropout](http://www.cs.toronto.edu/~rsalakhu/papers/srivastava14a.pdf)

[BatchNormalization](https://arxiv.org/pdf/1502.03167.pdf)